In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage
#basemessage is collection of all HumanMessage,AIMessage
from typing import TypedDict, Annotated
 

In [ ]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
       messages:Annotated[list[BaseMessage], add_messages]

In [ ]:
model=ChatOpenAI()

def chat_node(state:ChatState):
    #take user query from state
    messages=state['messages']
    #send to llm
    response=model.invoke(messages)
    #response store state
    return {'messages':[response]}

In [ ]:
checkpointer=checkpointer(MemorySaver)
graph=StateGraph(ChatState)
graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)
chatbot=graph.compile()
inital_state={
    'messages':[HumanMessage(content='What is the capital of India')]
}

chatbot.invoke(inital_state)




{'messages': [HumanMessage(content='What is the capital of India', additional_kwargs={}, response_metadata={}, id='ad4297b7-b1bc-48a0-b79d-a8a3ed2f8a13'),
  AIMessage(content='The capital of India is New Delhi.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 13, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EC3AeEREWSUfRj6enscdHEx2HnIyU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ff621-b287-7a11-9385-2b26de160df7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 8, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'c

In [ ]:
while True:
    user_message=input('TypeHere:')
    print("User:",user_message)

    if user_message.strip().lower() in ['quit','exit','bye']:
        break
    response=chatbot.invoke({'messages':[HumanMessage(content=user_message)]})

    print("AI Message: ",response['messages'][-1].content)

'''Here the problem with the chatbot is ,it doesn't hold any chat history where everytime it will
 invoke with fresh usermessage and every time is a fresh invoke 
 so we should use the persistence concept to have the chat history '''


User: Hi 
AI Message:  Hello! How can I assist you today?
User: How are you
AI Message:  I'm just a computer program, so I don't have feelings or emotions. But thank you for asking! How can I assist you today?
User: quit
